# Monte Carlo Methods for Option Pricing

From-scratch implementation of Monte Carlo simulation for pricing European, Asian, and barrier
options, with variance reduction techniques and convergence analysis.
This notebook builds everything from scratch using only NumPy and SciPy — no pricing libraries. Every formula is explained intuitively before being implemented.

> **Key Concept:** Monte Carlo is the "Swiss army knife" of derivatives pricing. When a closed-form formula doesn't exist (which is most of the time in practice), MC can still give you an answer — along with a confidence interval telling you how precise that answer is.

## Prerequisites and Learning Objectives

### What You Should Already Know

This notebook assumes familiarity with:
- **Basic probability:** random variables, expected values, the normal distribution, the Central Limit Theorem
- **Option basics:** what a call and put option are, the concept of "in the money" and "out of the money"
- **The Black-Scholes-Merton formula:** we use it as a benchmark (derived in a companion notebook)
- **Geometric Brownian Motion (GBM):** the standard model for stock prices (also covered in a companion notebook)

If you are unsure about any of these, review the relevant notebooks first. But do not worry --- we will explain every formula before using it.

### Learning Objectives

By the end of this notebook, you will be able to:
1. Explain why we simulate under the risk-neutral measure rather than the real-world measure
2. Implement Monte Carlo pricing for European, Asian, and barrier options from scratch
3. Quantify the accuracy of a Monte Carlo estimate using confidence intervals
4. Apply antithetic variates and control variates to reduce pricing errors
5. Verify barrier option prices using the in-out parity relationship
6. Determine when Monte Carlo is the appropriate pricing method versus analytical formulas or trees### Notation Used in This Notebook

| Symbol | Meaning |
|:-------|:--------|
| $S_0$ | Current stock price |
| $S_T$ | Stock price at time $T$ |
| $K$ | Strike price |
| $r$ | Risk-free interest rate (continuously compounded) |
| $\sigma$ | Volatility (annualised standard deviation of log-returns) |
| $T$ | Time to expiry (in years) |
| $N$ | Number of Monte Carlo paths |
| $Z$ | Standard normal random variable $Z \sim \mathcal{N}(0,1)$ |
| $E^Q[\cdot]$ | Expectation under the risk-neutral measure |


## 1. The Big Idea: Simulate Thousands of Possible Futures

Many financial derivatives have complex payoffs that depend on the entire path of the stock price, not just where it ends up. For example:

- **Asian options**: payoff depends on the *average* stock price over the option's life.
- **Barrier options**: the option is knocked out (becomes worthless) if the stock hits a certain level.
- **Lookback options**: payoff depends on the *maximum* or *minimum* stock price.

For these exotic options, there is no neat formula like Black-Scholes. But there is a simple and powerful alternative: **simulate the stock price many times and average the results**.

### The Analogy: Predicting Election Results

Imagine trying to predict an election outcome. You could:
1. Build a complex mathematical model of voter behavior.
2. Or... run thousands of simulated elections based on polling data, and see what fraction of them each candidate wins.

Monte Carlo option pricing works the same way:
1. Simulate thousands of possible stock price paths.
2. For each path, compute what the option would pay out.
3. Average the payouts and discount to today.

The more simulations you run, the more accurate your estimate becomes.

> **Key Concept:** Monte Carlo methods convert a hard analytical problem (computing an expectation) into a simple computational one (averaging random samples). They are the "brute force" of quantitative finance --- not elegant, but incredibly versatile. If you can write down the payoff, you can price it with Monte Carlo.

### When to Use Monte Carlo

| Situation | MC? | Why |
|-----------|:---:|-----|
| Vanilla European option | No | BSM formula is exact and instant |
| Path-dependent exotic | **Yes** | MC handles any payoff naturally |
| High-dimensional (basket of 50 stocks) | **Yes** | MC does not suffer from curse of dimensionality |
| American option | Maybe | Requires special techniques (Longstaff-Schwartz) |
| Need very high accuracy (< \$0.001) | Depends | MC is slow to converge; use variance reduction |### MC vs Other Pricing Methods

| Method | Best for | Limitations |
|:-------|:---------|:------------|
| **Analytical (BSM)** | European vanilla options | Very few products have closed forms |
| **Binomial/trinomial trees** | American options, 1-2 underlyings | Curse of dimensionality beyond 2-3 assets |
| **Finite differences** | 1D-2D PDEs, American exercise | Grid-based, doesn't scale beyond ~3 dimensions |
| **Monte Carlo** | Path-dependent, multi-asset, complex payoffs | Slow for American options, $O(1/\sqrt{N})$ convergence |

> **Key Concept:** Monte Carlo's unique strength is **scalability in dimension**. For a basket option on 50 stocks, trees and PDE methods are hopeless (the grid would have $50$ dimensions), but MC works just fine — each path simply has 50 correlated random variables instead of one.


### The Three-Step Monte Carlo Recipe

Every Monte Carlo option pricing problem follows the same three steps:

**Step 1: Simulate.** Generate random stock price paths under the risk-neutral measure (using drift $r$, not $\mu$).

**Step 2: Evaluate.** For each simulated path, compute the option payoff (e.g., $\max(S_T - K, 0)$ for a call).

**Step 3: Average and Discount.** Take the average of all payoffs and discount to today at the risk-free rate.

That is it. The entire method is an application of the law of large numbers: the sample average of many random payoffs converges to the true expected payoff.

The challenge is not the method itself --- it is making it fast enough and accurate enough to be useful in practice. The rest of this notebook addresses both challenges.

### Roadmap of This Notebook

Before we dive in, here is the journey we will take:

1. **Risk-neutral pricing** --- the theoretical foundation that justifies the Monte Carlo approach.
2. **GBM simulation** --- generating random stock price paths on a computer.
3. **European options** --- the simplest application, where we can validate against the exact BSM formula.
4. **Variance reduction** --- clever tricks to get accurate prices faster.
5. **Asian options** --- path-dependent options where MC truly shines.
6. **Barrier options** --- another path-dependent family with elegant mathematical properties.
7. **Convergence analysis** --- understanding the statistical properties of our estimates.

Each section builds on the previous one. By the end, you will have a complete Monte Carlo pricing toolkit.

> **Key Concept:** The power of Monte Carlo lies in its generality. Once you understand the basic framework (simulate paths, compute payoffs, average, discount), you can price virtually any derivative by simply changing the payoff function. The same code structure works for options on stocks, commodities, currencies, interest rates, and even multi-asset baskets.

## 2. Risk-Neutral Pricing: The Foundation

### Why Not Just Simulate with the Real Stock Return?

You might think: "I will simulate the stock with its expected return $\mu = 8\%$ and average the payoffs." This is wrong --- you would get the wrong price.

The problem is that different investors have different risk preferences. A risk-averse investor would pay less for the same uncertain payoff than a risk-neutral one. Which investor's price is "correct"?

### The Trick: Pretend Everyone Is Risk-Neutral

The fundamental theorem of asset pricing tells us something remarkable: to compute the correct option price, we can **pretend that everyone is risk-neutral** and use the risk-free rate $r$ instead of the real expected return $\mu$.

In this "risk-neutral world":
1. The stock grows at rate $r$ (not $\mu$) on average.
2. We compute expected payoffs using this modified distribution.
3. We discount at the risk-free rate $r$.

The result is the correct arbitrage-free price, regardless of anyone's actual risk preferences.

### The Risk-Neutral Pricing Formula

$$V_0 = e^{-rT}\, \mathbb{E}^{\mathbb{Q}}\bigl[\text{Payoff}(S_T)\bigr]$$

where:
- $V_0$ is today's option price
- $e^{-rT}$ discounts from expiry back to today at the risk-free rate
- $\mathbb{E}^{\mathbb{Q}}$ is the expectation under the risk-neutral measure (stock drift = $r$)

> **Key Concept:** Risk-neutral pricing is not an assumption about the world. The real world is full of risk-averse investors. But because options can be replicated by trading the underlying stock, their prices are the same as they would be in a risk-neutral world. This is the profound insight that makes derivative pricing possible.

### The Monte Carlo Approximation

Since we cannot compute the expectation analytically (especially for exotic payoffs), we approximate it:

$$\hat{V}_0 = e^{-rT} \frac{1}{N} \sum_{i=1}^{N} \text{Payoff}(S_T^{(i)})$$

where $S_T^{(1)}, S_T^{(2)}, \ldots, S_T^{(N)}$ are simulated terminal stock prices under the risk-neutral measure.

> **Common Mistake:** Using the real-world drift $\mu$ instead of the risk-free rate $r$ in simulation. This gives the expected payoff under the real-world measure, which is NOT the option price. Always simulate with drift $r$ for pricing.### The Fundamental Theorem in Plain English

The connection between real-world and risk-neutral pricing rests on one of the deepest results in mathematical finance:

> **Key Concept (Fundamental Theorem of Asset Pricing):** In a market with no arbitrage opportunities, there exists a probability measure (called the **risk-neutral** or **equivalent martingale** measure) under which all asset prices, discounted at the risk-free rate, are martingales (i.e., their expected future value equals their current value).

What does this mean practically?

1. **We don't need to know investors' risk preferences** to price derivatives
2. **We don't need to estimate the real-world drift $\mu$** — which is notoriously difficult
3. **We just replace $\mu$ with $r$** in our simulation, then discount payoffs at $r$

This is why two traders with very different views on where the stock is headed can still agree on the option price. The price depends on volatility and the risk-free rate, not on anyone's forecast of the stock's direction.

> **Common Mistake:** Students sometimes think risk-neutral pricing assumes investors are actually risk-neutral. They're not! The name is misleading. What it really means is: we've found a clever change of probability measure that lets us price as *if* everyone were risk-neutral, even though they aren't.


### A Concrete Example of Why Real-World Drift Is Wrong

Suppose Apple stock has $\mu = 12\%$ expected return and $\sigma = 30\%$ volatility. Two investors both agree on these numbers but disagree on the price of a call option:

- **Aggressive hedge fund:** Willing to accept more risk, values the option at $\$15$.
- **Conservative pension fund:** Demands a risk premium, values the option at $\$10$.

Neither is the "correct" price. The correct price is the one that prevents arbitrage --- and that turns out to be the risk-neutral price, computed using $r$ instead of $\mu$. Both investors must accept this price, because if it were different, someone could construct a riskless profit.

> **CFA Exam Tip:** Risk-neutral pricing is a foundational concept tested at both CFA Level I and Level II. The key insight is that the risk-neutral approach gives the same price as the no-arbitrage replication approach. You do NOT need to know an investor's risk aversion or the stock's expected return to price a derivative.

### Summary: The Risk-Neutral Pricing Framework

Let us consolidate the key ideas before moving to code:

| Step | What We Do | Why |
|------|-----------|-----|
| 1. Change the drift | Use $r$ instead of $\mu$ in the GBM | To price in the risk-neutral world |
| 2. Simulate paths | Generate $S_T^{(1)}, \ldots, S_T^{(N)}$ | Each represents one possible future |
| 3. Compute payoffs | $\text{Payoff}^{(i)} = \max(S_T^{(i)} - K, 0)$ for calls | The option's value at expiry on that path |
| 4. Average | $\bar{V} = \frac{1}{N}\sum_{i=1}^N \text{Payoff}^{(i)}$ | Law of large numbers gives the expectation |
| 5. Discount | $\hat{V}_0 = e^{-rT} \bar{V}$ | Time value of money |
| 6. Report SE | $\text{SE} = \text{std}(\text{payoffs}) / \sqrt{N}$ | Quantify the estimation error |

> **Key Concept:** The risk-neutral framework separates the problem into two independent parts: (1) modeling the stock price dynamics (the "physics"), and (2) evaluating the derivative payoff (the "contract"). Monte Carlo handles part (1) by simulation and part (2) by simple arithmetic. This modularity is what makes MC so flexible.

### Worked Example: Risk-Neutral Pricing by Hand

Before writing code, let's price a simple option by hand using the MC logic.

**Setup:** $S_0 = 100$, $r = 5\%$, $\sigma = 20\%$, $T = 1$ year, European call with $K = 105$.

**Step 1 — Simulate** four terminal prices using $S_T = 100 \times \exp[(0.05 - 0.02) + 0.20 \times Z]$:

| Path | $Z$ | $S_T$ | Payoff $\max(S_T - 105, 0)$ |
|:-----|:---:|:-----:|:---------------------------:|
| 1 | 0.52 | $100 e^{0.134} = 114.34$ | 9.34 |
| 2 | −1.13 | $100 e^{-0.196} = 82.20$ | 0.00 |
| 3 | 1.87 | $100 e^{0.404} = 149.86$ | 44.86 |
| 4 | −0.41 | $100 e^{-0.052} = 94.94$ | 0.00 |

**Step 2 — Average:** Mean payoff $= (9.34 + 0 + 44.86 + 0)/4 = 13.55$

**Step 3 — Discount:** Call price $\approx e^{-0.05} \times 13.55 = 12.89$

With only 4 paths, this is rough. The true BSM price is ~\$8.02. But with 100,000 paths, the MC estimate converges to the analytical value.

> **Common Mistake:** Never simulate with the real-world drift $\mu$. Always use $r$ in the exponent. Using $\mu$ gives you an expected *future payoff*, not today's *fair price*.

## 3. Simulating Stock Price Paths

Under the risk-neutral measure, the stock price follows GBM with drift $r$:

$$S_T = S_0 \exp\!\left[\left(r - \frac{\sigma^2}{2}\right)T + \sigma\sqrt{T}\,Z\right], \quad Z \sim N(0,1)$$

For **European options** (payoff depends only on the final price), we need just the terminal value --- one random number per path.

For **path-dependent options** (Asian, barrier, lookback), we need the full path. We discretize time into $M$ steps and simulate step by step:

$$S_{t+\Delta t} = S_t \exp\!\left[\left(r - \frac{\sigma^2}{2}\right)\Delta t + \sigma\sqrt{\Delta t}\,Z_t\right]$$

where each $Z_t$ is an independent standard normal random variable.

> **Key Concept:** The $\sigma^2/2$ correction (the Ito correction) ensures that $\mathbb{E}[S_T] = S_0 e^{rT}$ --- the stock grows at the risk-free rate on average under the risk-neutral measure. Without this correction, you would get a systematic upward bias.

The code below implements both single-step (for European) and multi-step (for exotic) simulation.

### Understanding the Ito Correction Term

The $\sigma^2/2$ term puzzles many beginners. Here is the intuition: when you take the exponential of a normal random variable, the result is *not* centered where you might expect. The exponential function is convex, so by Jensen's inequality, $\mathbb{E}[e^X] > e^{\mathbb{E}[X]}$. The $\sigma^2/2$ correction exactly compensates for this convexity effect.

Without this correction:
- $\mathbb{E}[S_T]$ would be $S_0 e^{(r + \sigma^2/2)T}$ instead of $S_0 e^{rT}$
- The stock would grow faster than the risk-free rate on average
- This would violate the no-arbitrage condition

### Worked Example: Simulating One Path by Hand

Let $S_0 = 100$, $r = 5\%$, $\sigma = 20\%$, $T = 1$ year, $\Delta t = 0.25$ (quarterly steps).

Suppose we draw $Z_1 = 0.5$, $Z_2 = -1.2$, $Z_3 = 0.8$, $Z_4 = 0.1$.

**Step 1** ($t = 0$ to $t = 0.25$):
$$S_{0.25} = 100 \times \exp\bigl[(0.05 - 0.02)\times 0.25 + 0.20 \times \sqrt{0.25} \times 0.5\bigr] = 100 \times e^{0.0575} = 105.92$$

**Step 2** ($t = 0.25$ to $t = 0.50$):
$$S_{0.50} = 105.92 \times \exp\bigl[0.0075 + 0.10 \times (-1.2)\bigr] = 105.92 \times e^{-0.1125} = 94.65$$

**Step 3** ($t = 0.50$ to $t = 0.75$):
$$S_{0.75} = 94.65 \times \exp\bigl[0.0075 + 0.10 \times 0.8\bigr] = 94.65 \times e^{0.0875} = 103.30$$

**Step 4** ($t = 0.75$ to $t = 1.00$):
$$S_{1.00} = 103.30 \times \exp\bigl[0.0075 + 0.10 \times 0.1\bigr] = 103.30 \times e^{0.0175} = 105.12$$

This is one simulated path. Repeat thousands of times to build up a distribution of terminal stock prices.### A Numerical Check of the Correction

If we omit the $-\sigma^2/2$ term and simulate $S_T = S_0 e^{rT + \sigma\sqrt{T}Z}$, the average terminal price would be:

$$E[S_T] = S_0 e^{rT} \cdot E[e^{\sigma\sqrt{T}Z}] = S_0 e^{rT} \cdot e^{\sigma^2 T/2} = S_0 e^{(r + \sigma^2/2)T}$$

This is **too high** — the stock grows faster than $r$, violating risk-neutral pricing. The $-\sigma^2/2$ correction fixes this:

$$E[S_T] = S_0 e^{(r - \sigma^2/2)T} \cdot e^{\sigma^2 T/2} = S_0 e^{rT} \quad \checkmark$$

> **Key Concept:** The Itô correction $-\sigma^2/2$ is not an approximation or a modelling choice — it is a mathematical consequence of applying the exponential function to a stochastic process. Without it, your simulation will systematically overprice everything.


In [ ]:
%matplotlib inline
import numpy as np
from scipy import stats, optimize
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
rng = np.random.default_rng(SEED)

ATOL = 1e-10
RTOL = 1e-6

PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'gold'
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

Let's now implement GBM simulation in Python. The function below simulates terminal stock prices and full paths under the risk-neutral measure. Watch how each line maps directly to the formula above:

In [ ]:
def simulate_gbm_terminal(S0, r, sigma, T, n_paths, rng):
    """Simulate terminal stock prices under risk-neutral measure (single step).
    
    This is all we need for European options: just the final price.
    """
    Z = rng.normal(size=n_paths)
    ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    return ST, Z


def simulate_gbm_paths(S0, r, sigma, T, n_steps, n_paths, rng):
    """Simulate full GBM paths under risk-neutral measure.
    
    Needed for path-dependent options (Asian, barrier, lookback).
    
    Returns
    -------
    t : array (n_steps+1,)
    S : array (n_paths, n_steps+1)
    """
    dt = T / n_steps
    t = np.linspace(0, T, n_steps + 1)
    
    # Generate all random shocks at once (vectorized for speed)
    Z = rng.normal(size=(n_paths, n_steps))
    increments = (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
    
    # Build paths via cumulative sum of log-returns
    log_S = np.zeros((n_paths, n_steps + 1))
    log_S[:, 0] = np.log(S0)
    log_S[:, 1:] = np.log(S0) + np.cumsum(increments, axis=1)
    
    S = np.exp(log_S)
    return t, S


# Visualize some paths
S0, r, sigma, T = 100, 0.05, 0.2, 1.0
t, S = simulate_gbm_paths(S0, r, sigma, T, n_steps=252, n_paths=20, rng=rng)

fig, ax = plt.subplots(figsize=(10, 6))
for i in range(20):
    ax.plot(t, S[i], alpha=0.5, linewidth=0.8)
ax.axhline(S0, color='black', linestyle='--', alpha=0.5, label=f'$S_0 = {S0}$')
ax.set_xlabel('Time (years)')
ax.set_ylabel('Stock Price')
ax.set_title('GBM Paths under Risk-Neutral Measure')
ax.legend()
plt.tight_layout()
plt.show()

Each line is one possible future for the stock. Under the risk-neutral measure, the expected growth rate is the risk-free rate $r = 5\%$, but individual paths can wander far above or below. The option price is the average payoff across all these scenarios.

Notice the fan-like shape: paths are bunched together near $t = 0$ and spread out over time. This reflects the growing uncertainty --- the longer we look into the future, the wider the range of possible outcomes. The *width* of the fan is determined by $\sigma$; higher volatility means a wider fan.

### Reading the Path Plot

Each coloured line represents one possible future for the stock under the risk-neutral measure. Key observations:

- **Paths fan out over time** — uncertainty grows with $\sigma\sqrt{T}$
- **Some end far above $S_0$, others far below** — this is the lognormal distribution at work
- **The average terminal price $\approx S_0 e^{rT}$** — not $S_0 e^{\mu T}$, because we simulate under the risk-neutral measure

> **Key Concept:** Under risk-neutral pricing, the expected growth rate of the stock equals the risk-free rate $r$. This doesn't mean we believe stocks only earn the risk-free rate — it's a mathematical device that lets us price derivatives by simple discounting.

## 4. Pricing European Options

Let us start with the simplest case: a European call option.

The payoff at expiry is $\max(S_T - K, 0)$. In words: if the stock finishes above the strike, you earn the difference. Otherwise, you get nothing.

The Monte Carlo price is:

$$\hat{C} = e^{-rT} \frac{1}{N} \sum_{i=1}^{N} \max(S_T^{(i)} - K, 0)$$

### Worked Example (by hand, then code)

Suppose we simulate just 5 paths with $S_0 = 100$, $K = 100$, and get terminal prices: $\$112$, $\$87$, $\$105$, $\$93$, $\$118$.

Payoffs: $\max(112-100,0)=12$, $\max(87-100,0)=0$, $\max(105-100,0)=5$, $\max(93-100,0)=0$, $\max(118-100,0)=18$.

Average payoff: $(12 + 0 + 5 + 0 + 18)/5 = 7.00$.

Discounted price: $e^{-0.05} \times 7.00 = 6.66$.

With only 5 paths, this is a rough estimate. The BSM formula gives $\$10.45$. We need many more paths.

> **Key Concept:** The standard error of the MC estimate is $\sigma_{\text{payoff}} / \sqrt{N}$, where $\sigma_{\text{payoff}}$ is the standard deviation of the discounted payoffs. To halve the error, you need 4x more paths. To get one more decimal place, you need 100x more paths. This $O(1/\sqrt{N})$ convergence is the fundamental limitation of Monte Carlo.

### Understanding the Standard Error

The standard error formula follows from the Central Limit Theorem. If each payoff has standard deviation $\sigma_{\text{payoff}}$, then the average of $N$ independent payoffs has standard deviation $\sigma_{\text{payoff}}/\sqrt{N}$.

This gives us a **95% confidence interval** for the true option price:

$$\hat{V} \pm 1.96 \times \text{SE}$$

For example, if our MC estimate is $\$10.50$ with $\text{SE} = \$0.04$, then we are 95% confident the true price is in the interval $[\$10.42, \$10.58]$.

> **CFA Exam Tip:** The CFA exam tests the relationship between sample size and precision. Remember: doubling precision (halving the confidence interval width) requires quadrupling the sample size. This applies to Monte Carlo simulation and to any sampling-based estimate.

In [ ]:
def bsm_call(S, K, r, T, sigma):
    """Analytical BSM call price for reference."""
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * stats.norm.cdf(d1) - K * np.exp(-r * T) * stats.norm.cdf(d2)


def bsm_put(S, K, r, T, sigma):
    """Analytical BSM put price for reference."""
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return K * np.exp(-r * T) * stats.norm.cdf(-d2) - S * stats.norm.cdf(-d1)


def mc_european(S0, K, r, T, sigma, n_paths, option_type='call', rng=rng):
    """Monte Carlo European option price with standard error."""
    ST, _ = simulate_gbm_terminal(S0, r, sigma, T, n_paths, rng)
    
    if option_type == 'call':
        payoffs = np.maximum(ST - K, 0)  # call: max(S_T - K, 0)
    else:
        payoffs = np.maximum(K - ST, 0)  # put: max(K - S_T, 0)
    
    disc_payoffs = np.exp(-r * T) * payoffs  # discount to present value
    price = disc_payoffs.mean()               # average = MC estimate
    se = disc_payoffs.std() / np.sqrt(n_paths)  # standard error
    
    return price, se

### The Computational Cost of Accuracy

The following table puts the $O(1/\sqrt{N})$ convergence into practical terms:

| Desired Accuracy | Required Paths | Approximate Compute Time* |
|-----------------|----------------|--------------------------|
| $\pm \$1.00$ (very rough) | ~1,000 | Milliseconds |
| $\pm \$0.10$ (decent) | ~100,000 | Seconds |
| $\pm \$0.01$ (good for trading) | ~10,000,000 | Minutes |
| $\pm \$0.001$ (benchmark quality) | ~1,000,000,000 | Hours |

*For a single European option on a modern CPU. Path-dependent options with many time steps are proportionally slower.

This table explains why variance reduction is not a luxury --- it is a necessity for production pricing. A 10x variance reduction effectively gives you the accuracy of 10x more paths for free.

Now let us see how the MC estimate improves as we increase the number of paths.

In [ ]:
# Compare MC vs BSM for increasing number of paths
K = 100
bsm_ref = bsm_call(S0, K, r, T, sigma)

print(f"BSM analytical call price: {bsm_ref:.6f}\n")
print(f"{'N paths':>12s} {'MC Price':>12s} {'Std Error':>12s} {'Error':>12s}")
print("-" * 52)

for n in [1000, 10000, 100000, 1000000]:
    price, se = mc_european(S0, K, r, T, sigma, n, 'call', rng=np.random.default_rng(42))
    print(f"{n:12d} {price:12.6f} {se:12.6f} {abs(price - bsm_ref):12.6f}")

Notice the pattern:

| Paths | Standard Error | Approximate Accuracy |
|-------|---------------|---------------------|
| 1,000 | ~\$0.40 | Rough estimate |
| 10,000 | ~\$0.13 | Decent |
| 100,000 | ~\$0.04 | Good |
| 1,000,000 | ~\$0.013 | Very good |

Each 10x increase in paths reduces the error by about $\sqrt{10} \approx 3.16$ times. This is the $O(1/\sqrt{N})$ convergence rate --- fundamentally slow.

> **Key Concept:** To get one more decimal place of accuracy, you need 100x more paths. This motivates variance reduction techniques, which we explore next. The goal: make each path "count for more" so we need fewer of them.

> **Important:** The standard error gives us a confidence interval. With 100,000 paths and SE = \$0.04, the 95% CI is approximately $\hat{V} \pm 1.96 \times 0.04 = \hat{V} \pm \$0.08$. Always report confidence intervals with MC prices --- a point estimate without error bounds is meaningless.

Let us visualize the convergence.

### What This Means in Practice

Consider a trading desk that needs to price 10,000 exotic options per day, each requiring 100,000 paths with 252 time steps. That is $10{,}000 \times 100{,}000 \times 252 = 252$ billion random number generations per day. This is why quantitative finance firms invest heavily in computational infrastructure (GPUs, FPGAs, and cluster computing) for Monte Carlo.

> **CFA Exam Tip:** The Monte Carlo standard error is $SE = \hat{\sigma} / \sqrt{N}$, where $\hat{\sigma}$ is the standard deviation of the simulated payoffs. To **halve** the error, you need **4x** as many paths. This square-root relationship is fundamental.

In [ ]:
# Convergence plot
n_paths_range = np.logspace(2, 6, 50).astype(int)
mc_prices = []
mc_ses = []

for n in n_paths_range:
    p, se = mc_european(S0, K, r, T, sigma, n, 'call', rng=np.random.default_rng(42))
    mc_prices.append(p)
    mc_ses.append(se)

mc_prices = np.array(mc_prices)
mc_ses = np.array(mc_ses)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].semilogx(n_paths_range, mc_prices, color=PRIMARY, linewidth=1)
axes[0].fill_between(n_paths_range, mc_prices - 1.96*mc_ses, mc_prices + 1.96*mc_ses,
                     alpha=0.2, color=PRIMARY)
axes[0].axhline(bsm_ref, color=SECONDARY, linestyle='--', linewidth=2, label=f'BSM = {bsm_ref:.4f}')
axes[0].set_xlabel('Number of Paths')
axes[0].set_ylabel('MC Price')
axes[0].set_title('MC Convergence (95% CI)')
axes[0].legend()

axes[1].loglog(n_paths_range, mc_ses, color=PRIMARY, linewidth=2, label='MC Std Error')
ref = mc_ses[0] * np.sqrt(n_paths_range[0]) / np.sqrt(n_paths_range)
axes[1].loglog(n_paths_range, ref, color='gray', linestyle='--', alpha=0.7, label=r'$O(1/\sqrt{N})$')
axes[1].set_xlabel('Number of Paths')
axes[1].set_ylabel('Standard Error')
axes[1].set_title('MC Standard Error')
axes[1].legend()

plt.tight_layout()
plt.show()

The left plot shows the MC estimate (blue) converging to the BSM value (orange dashed) as we add more paths. The shaded band is the 95% confidence interval, which narrows steadily.

The right plot (log-log scale) shows that the standard error exactly follows the $O(1/\sqrt{N})$ slope --- the theoretical rate.

### Why $O(1/\sqrt{N})$ Matters in Practice

The convergence rate $O(1/\sqrt{N})$ means Monte Carlo is inherently slow for high precision:

| Desired accuracy | Paths needed (approx) | Time at 1M paths/sec |
|:-----------------|:---------------------:|:--------------------:|
| 2 decimal places | 10,000 | 0.01 sec |
| 3 decimal places | 1,000,000 | 1 sec |
| 4 decimal places | 100,000,000 | 100 sec |

To get one more decimal place of accuracy, you need **100x more paths**. This motivates variance reduction: instead of brute-forcing more paths, use clever tricks to get the same accuracy with fewer simulations.

> **Important:** Despite this slow convergence, MC has one killer advantage: **the convergence rate doesn't depend on dimension**. Pricing an option on 100 correlated assets takes essentially the same effort per path as pricing one on a single asset. This is why MC dominates for multi-asset derivatives.

### Other Variance Reduction Techniques (Brief Mention)

While we implement antithetic variates and control variates in this notebook, there are several other techniques used in practice:

| Technique | Idea | When It Helps |
|-----------|------|---------------|
| **Importance sampling** | Sample more from regions that contribute most to the expectation | Deep OTM options where most paths have zero payoff |
| **Stratified sampling** | Divide the random number space into strata and sample from each | General improvement, especially for tail-dependent payoffs |
| **Quasi-Monte Carlo** | Replace pseudo-random numbers with low-discrepancy sequences (Sobol, Halton) | High-dimensional problems; can improve rate to $O(1/N)$ |
| **Moment matching** | Adjust random draws so sample moments match theoretical moments | Ensures the simulation satisfies known constraints exactly |

In production systems, combinations of these techniques are used. A typical setup might use Sobol sequences with antithetic variates and control variates simultaneously.

## 5. Variance Reduction: Getting More with Less

The $O(1/\sqrt{N})$ convergence is painfully slow. Variance reduction techniques make each path "count for more," effectively multiplying the number of useful samples without actually simulating more paths.

### Technique 1: Antithetic Variates

**The idea:** When you simulate a path using random numbers $Z$, also simulate a "mirror" path using $-Z$. These two paths are negatively correlated, so their average has lower variance than two independent paths.

**Analogy:** Instead of asking two random people for directions, ask the same person to give directions from two opposite starting points. The errors tend to cancel.

**Why it works mathematically:** If $f(Z)$ and $f(-Z)$ are negatively correlated, then $\text{Var}\!\left(\frac{f(Z) + f(-Z)}{2}\right) < \text{Var}(f(Z))$.

### Technique 2: Control Variates

**The idea:** Find a related quantity whose expectation you know exactly. Use it to "correct" your estimate.

**Analogy:** Imagine you are estimating the average height of people in a room, but your measuring tape is slightly off. If you also measure a person whose height you already know (the "control"), you can calibrate your tape and correct all measurements.

For option pricing, we know that $\mathbb{E}^{\mathbb{Q}}[e^{-rT} S_T] = S_0$ exactly (discounted stock price is a martingale). We can use this as a control variate to adjust our option price estimate.

**The formula:**

$$\hat{V}_{\text{adjusted}} = \hat{V}_{\text{crude}} - \beta\left(\bar{X} - \mathbb{E}[X]\right)$$

where $X$ is the control variate and $\beta$ is chosen to minimize variance (the optimal $\beta$ is $\text{Cov}(V, X) / \text{Var}(X)$).

> **Key Concept:** Variance reduction is the art of using structure in the problem to reduce randomness. The more correlated your control variate is with the option payoff, the larger the variance reduction. For a call option, the discounted stock price is a natural control because high stock prices lead to both high payoffs and high control values.### Antithetic Variates: The Mirror Trick

The idea is simple and elegant. For every random draw $Z$, also use $-Z$. Since $Z$ and $-Z$ are both standard normal, both are valid samples. But they are **negatively correlated** — when one produces a high stock price, the other produces a low one.

**Why does this help?** Consider the variance of an average:

$$\text{Var}\left(\frac{X_1 + X_2}{2}\right) = \frac{\text{Var}(X_1) + \text{Var}(X_2) + 2\text{Cov}(X_1, X_2)}{4}$$

If $X_1$ and $X_2$ are independent, the covariance is zero. But if they're negatively correlated (as with antithetic pairs), the covariance is *negative*, which **reduces** the overall variance. Free improvement!

### Control Variates: Borrowing from a Known Answer

The idea: suppose you want to estimate $E[f(X)]$ but you also know the exact value of $E[g(X)]$ for some related function $g$. Then:

$$\hat{\mu}_{\text{CV}} = \frac{1}{N}\sum f(X_i) - c\left(\frac{1}{N}\sum g(X_i) - E[g(X)]\right)$$

The correction term adjusts for the fact that your sample average of $g$ differs from its known mean. If $f$ and $g$ are correlated, this adjustment also corrects $f$.

**In option pricing:** We use the delta hedge P&L or geometric average option (which has a closed form) as the control. The optimal coefficient $c$ is chosen to minimise variance:

$$c^* = \frac{\text{Cov}(f, g)}{\text{Var}(g)}$$

> **Key Concept:** Control variates can reduce variance by a factor of 10x or more when the control is highly correlated with the target. This is the single most effective variance reduction technique in practice.


### Deeper Dive: Antithetic Variates Worked Example

Suppose we draw $Z = 0.8$. Then:
- **Original path:** $S_T = 100 \times e^{(0.05 - 0.02) \times 1 + 0.20 \times 0.8} = 100 \times e^{0.19} = 120.92$
- **Mirror path:** $S_T = 100 \times e^{(0.05 - 0.02) \times 1 + 0.20 \times (-0.8)} = 100 \times e^{-0.13} = 87.81$
- **Call payoffs** ($K = 100$): $\max(120.92 - 100, 0) = 20.92$ and $\max(87.81 - 100, 0) = 0$
- **Antithetic estimate for this pair:** $(20.92 + 0)/2 = 10.46$

The high payoff from the $Z$ path is "balanced" by the low payoff from the $-Z$ path, reducing the overall variability of the estimate.

### Deeper Dive: Control Variates Intuition

Think of control variates as a "calibration" step. If in a particular simulation the discounted stock price $\bar{X}$ came out higher than its known mean $S_0$, then the payoffs probably also came out higher than their true average. The correction $-\beta(\bar{X} - S_0)$ adjusts for this "luck of the draw."

The optimal $\beta$ measures how strongly correlated the payoffs are with the control. Higher correlation means a larger correction and greater variance reduction.

> **CFA Exam Tip:** The CFA curriculum mentions antithetic variates and control variates as techniques to improve Monte Carlo efficiency. You should know that they reduce the standard error without increasing the number of simulations, and you should understand the basic intuition behind each.

Let's implement both variance reduction methods and compare them head-to-head. The code computes prices using standard MC, antithetic variates, and control variates on the same problem.

> **What to watch for:** The standard errors should be noticeably smaller for antithetic and control variate methods, even with the same number of paths.

In [ ]:
def mc_antithetic(S0, K, r, T, sigma, n_paths, option_type='call', rng=rng):
    """MC pricing with antithetic variates.
    
    For each random draw Z, we also use -Z (the mirror path).
    """
    n_half = n_paths // 2
    Z = rng.normal(size=n_half)
    
    # Original path and mirror path
    ST_plus  = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    ST_minus = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*(-Z))
    
    if option_type == 'call':
        payoffs = 0.5 * (np.maximum(ST_plus - K, 0) + np.maximum(ST_minus - K, 0))
    else:
        payoffs = 0.5 * (np.maximum(K - ST_plus, 0) + np.maximum(K - ST_minus, 0))
    
    disc_payoffs = np.exp(-r * T) * payoffs
    return disc_payoffs.mean(), disc_payoffs.std() / np.sqrt(n_half)


def mc_control_variate(S0, K, r, T, sigma, n_paths, option_type='call', rng=rng):
    """MC pricing with control variate (using discounted S_T as control).
    
    We know E[e^{-rT} S_T] = S_0 exactly, so we can use this to
    correct our estimate.
    """
    ST, Z = simulate_gbm_terminal(S0, r, sigma, T, n_paths, rng)
    
    if option_type == 'call':
        payoffs = np.maximum(ST - K, 0)
    else:
        payoffs = np.maximum(K - ST, 0)
    
    disc_payoffs = np.exp(-r * T) * payoffs
    
    # Control variate: discounted S_T, with known mean = S0
    control = np.exp(-r * T) * ST
    control_mean = S0
    
    # Optimal beta minimizes variance of the adjusted estimator
    cov_matrix = np.cov(disc_payoffs, control)
    beta = cov_matrix[0, 1] / cov_matrix[1, 1]
    
    # Adjusted estimator: subtract the control's deviation from its known mean
    adjusted = disc_payoffs - beta * (control - control_mean)
    return adjusted.mean(), adjusted.std() / np.sqrt(n_paths)

Let us compare all three methods on the same problem. The "Variance Ratio" column tells us the effective speedup factor --- a ratio of 5 means "this method achieves the same accuracy as standard MC with 5x as many paths."

In [ ]:
# Compare methods
n = 100000
seed = 42

p_std, se_std = mc_european(S0, K, r, T, sigma, n, 'call', rng=np.random.default_rng(seed))
p_av, se_av   = mc_antithetic(S0, K, r, T, sigma, n, 'call', rng=np.random.default_rng(seed))
p_cv, se_cv   = mc_control_variate(S0, K, r, T, sigma, n, 'call', rng=np.random.default_rng(seed))

print(f"BSM reference: {bsm_ref:.6f}\n")
print(f"{'Method':>20s} {'Price':>10s} {'SE':>10s} {'Var Ratio':>10s}")
print("-" * 55)
print(f"{'Standard MC':>20s} {p_std:10.6f} {se_std:10.6f} {1.0:10.2f}")
print(f"{'Antithetic':>20s} {p_av:10.6f} {se_av:10.6f} {(se_std/se_av)**2:10.2f}")
print(f"{'Control Variate':>20s} {p_cv:10.6f} {se_cv:10.6f} {(se_std/se_cv)**2:10.2f}")

The **Variance Ratio** column shows how many times more efficient each method is compared to standard MC. A ratio of 3 means "this method is equivalent to using 3x as many paths with standard MC."

- **Antithetic variates** typically give a modest improvement (1.5x--3x).
- **Control variates** can give substantial improvements (3x--10x or more), depending on how correlated the control is with the payoff.

In practice, combining both techniques (and others, like importance sampling) is common for production-quality pricing.

> **Important:** Variance reduction does not change the $O(1/\sqrt{N})$ convergence rate --- it reduces the constant. The lines on a log-log plot have the same slope but start at different heights.

### When to Use Which Technique

| Technique | Best For | Not Great For |
|-----------|----------|---------------|
| Antithetic variates | Monotone payoffs (calls, puts) | Digital/binary options (discontinuous payoffs) |
| Control variates | When a correlated analytical price exists | Truly exotic payoffs with no known relatives |
| Both combined | General-purpose improvement | - |

In [ ]:
# Efficiency comparison across sample sizes
n_range = np.logspace(2, 5, 30).astype(int)
ses = {'Standard': [], 'Antithetic': [], 'Control Variate': []}

for n in n_range:
    _, se = mc_european(S0, K, r, T, sigma, n, 'call', rng=np.random.default_rng(42))
    ses['Standard'].append(se)
    _, se = mc_antithetic(S0, K, r, T, sigma, n, 'call', rng=np.random.default_rng(42))
    ses['Antithetic'].append(se)
    _, se = mc_control_variate(S0, K, r, T, sigma, n, 'call', rng=np.random.default_rng(42))
    ses['Control Variate'].append(se)

fig, ax = plt.subplots(figsize=(10, 6))
colors = {'Standard': PRIMARY, 'Antithetic': SECONDARY, 'Control Variate': TERTIARY}
for method, se_vals in ses.items():
    ax.loglog(n_range, se_vals, color=colors[method], linewidth=2, label=method)

ax.set_xlabel('Number of Paths')
ax.set_ylabel('Standard Error')
ax.set_title('Variance Reduction Comparison')
ax.legend()
plt.tight_layout()
plt.show()

All three lines have the same slope ($-1/2$ on log-log, i.e., $O(1/\sqrt{N})$), but the variance-reduced methods start at a lower level. The vertical gap between lines represents the efficiency gain.

To read the chart: at any given number of paths, the control variate method achieves a standard error that the standard method would need many more paths to match. For example, if the control variate with 10,000 paths has the same SE as standard MC with 50,000 paths, the variance ratio is 5.

> **Key Concept:** Variance reduction doesn't change the $O(1/\sqrt{N})$ convergence *rate* — all three lines have the same slope on the log-log plot. What it does is shift the line *downward*, giving you the same accuracy with fewer paths. Control variates are typically the most effective when a good control is available (like the BSM delta).

## 6. Asian Options: Payoffs Based on Averages

### What Is an Asian Option?

An **Asian option** pays off based on the *average* stock price over the option's life, rather than the final price.

Why would anyone want this? Two main reasons:

1. **Hedging ongoing exposure**: A company that buys oil continuously throughout the year cares about the average price, not the price on one particular day.
2. **Cheaper than vanilla**: Averaging reduces the volatility of the payoff, so Asian options are cheaper than standard options.

### Types of Asian Options

| Type | Average | Payoff (call) |
|------|---------|--------------|
| Arithmetic average | $A = \frac{1}{M}\sum_{i=1}^{M} S_{t_i}$ | $\max(A - K, 0)$ |
| Geometric average | $G = \left(\prod_{i=1}^{M} S_{t_i}\right)^{1/M}$ | $\max(G - K, 0)$ |

The geometric average has a closed-form price (useful for validation), but the arithmetic average --- which is more common in practice --- does not. This is a perfect use case for Monte Carlo.

> **Key Concept:** The arithmetic mean is always greater than or equal to the geometric mean (AM-GM inequality). Therefore, arithmetic Asian options are always worth at least as much as geometric Asian options. Both are worth less than vanilla European options because averaging reduces the effective volatility of the payoff.

### Why Monte Carlo Is Natural Here

To compute the average, we need to know the stock price at multiple points in time --- we need the full path, not just the terminal value. Monte Carlo naturally produces full paths, so pricing Asian options is straightforward: simulate paths, compute the average along each path, compute the payoff, and average.

### Practical Considerations for Asian Options

**Monitoring frequency matters.** An Asian option monitored daily (252 observations per year) behaves differently from one monitored weekly (52 observations) or monthly (12 observations). More frequent monitoring means the average tracks the continuous path more closely, generally making the option cheaper (because the average is even smoother).

**Start date matters.** Some Asian options begin averaging from day one ("full-term average"), while others begin averaging partway through the option's life ("partial average"). A partial average Asian option is more expensive because the averaging period is shorter, reducing less of the volatility.

**Real-world example:** Many commodity contracts in Asia (hence the name "Asian option") use monthly averaging. An oil company might buy an Asian call on crude oil with monthly averaging over the calendar year. The payoff depends on the average of 12 monthly settlement prices rather than any single price.

### Real-World Example: The Airline Fuel Buyer

Southwest Airlines buys jet fuel every week throughout the year. Their true exposure is to the *average* fuel price over the year, not the price on any single day. An Asian call option on jet fuel with the average price as the underlying perfectly matches their hedging need. A vanilla European option would provide protection against the year-end price, which might be very different from the average price they actually paid.

### Why Can We Not Solve the Arithmetic Asian Analytically?

The geometric average of lognormal random variables is itself lognormal (because the log of the geometric mean is the arithmetic mean of the logs, which is normal). So the geometric Asian has a known distribution and a closed-form price.

The arithmetic average of lognormal random variables, however, is NOT lognormal. Its distribution has no known closed form. This is why we must resort to Monte Carlo (or other numerical methods) for arithmetic Asian options.

> **CFA Exam Tip:** Asian options appear in the CFA Level II derivatives curriculum. Know that they are path-dependent, that the arithmetic version is more common but harder to price, and that they are cheaper than vanilla options because averaging reduces effective volatility.### Arithmetic vs Geometric Average: When Does It Matter?

The **arithmetic** average is the simple mean: $\bar{S} = \frac{1}{M}\sum S_{t_j}$

The **geometric** average is the product-based mean: $\bar{S}_G = \left(\prod S_{t_j}\right)^{1/M}$

By the AM-GM inequality, $\bar{S} \geq \bar{S}_G$ always. This means:
- Arithmetic average calls are **more expensive** than geometric average calls
- Arithmetic average puts are **cheaper** than geometric average puts

The geometric average option has a **closed-form** solution (because the product of lognormals is lognormal), making it useful as a **control variate** for pricing the more common arithmetic Asian option.

> **CFA Exam Tip:** In practice, nearly all Asian options use the arithmetic average, because that's what matters for actual hedging (e.g., average fuel cost over a quarter). The geometric average is a mathematical convenience, not a product people trade.


Let's implement Monte Carlo pricing for Asian options. The key difference from European pricing is that we need to simulate the **entire path** (not just the terminal value) to compute the running average:

> **Implementation note:** We simulate `n_steps` price points along each path, compute the average (arithmetic or geometric), and then calculate the payoff from that average.

In [ ]:
def mc_asian(S0, K, r, T, sigma, n_steps, n_paths, average_type='arithmetic',
             option_type='call', rng=rng):
    """Price an Asian option via Monte Carlo.
    
    Simulates full paths and computes the average price along each path.
    """
    t, S = simulate_gbm_paths(S0, r, sigma, T, n_steps, n_paths, rng)
    
    # Monitor prices at each time step (exclude t=0)
    S_monitor = S[:, 1:]
    
    if average_type == 'arithmetic':
        A = S_monitor.mean(axis=1)  # simple average
    elif average_type == 'geometric':
        A = np.exp(np.log(S_monitor).mean(axis=1))  # geometric mean
    else:
        raise ValueError(f"Unknown average type: {average_type}")
    
    if option_type == 'call':
        payoffs = np.maximum(A - K, 0)
    else:
        payoffs = np.maximum(K - A, 0)
    
    disc_payoffs = np.exp(-r * T) * payoffs
    return disc_payoffs.mean(), disc_payoffs.std() / np.sqrt(n_paths)


def geometric_asian_closed_form(S0, K, r, T, sigma, n_steps):
    """Closed-form price for geometric Asian call (discrete monitoring)."""
    dt = T / n_steps
    sigma_a = sigma * np.sqrt((2 * n_steps + 1) / (6 * (n_steps + 1)))
    rho = 0.5 * (r - 0.5 * sigma**2 + sigma_a**2)
    
    d1 = (np.log(S0 / K) + (rho + 0.5 * sigma_a**2) * T) / (sigma_a * np.sqrt(T))
    d2 = d1 - sigma_a * np.sqrt(T)
    
    price = np.exp(-r * T) * (S0 * np.exp(rho * T) * stats.norm.cdf(d1) - K * stats.norm.cdf(d2))
    return price


# Price Asian options
n_steps = 252  # daily monitoring
n_paths = 200000

arith_price, arith_se = mc_asian(S0, K, r, T, sigma, n_steps, n_paths,
                                  'arithmetic', 'call', rng=np.random.default_rng(42))
geom_price, geom_se = mc_asian(S0, K, r, T, sigma, n_steps, n_paths,
                                'geometric', 'call', rng=np.random.default_rng(42))
geom_closed = geometric_asian_closed_form(S0, K, r, T, sigma, n_steps)

print(f"Asian Call Prices (K={K}, T={T}, sigma={sigma})")
print(f"  Arithmetic (MC): {arith_price:.4f} +/- {arith_se:.4f}")
print(f"  Geometric (MC):  {geom_price:.4f} +/- {geom_se:.4f}")
print(f"  Geometric (CF):  {geom_closed:.4f}")
print(f"  European (BSM):  {bsm_call(S0, K, r, T, sigma):.4f}")
print(f"\nNote: Asian < European because averaging reduces volatility.")

As expected:
- The Asian options are cheaper than the European option (averaging reduces effective volatility).
- The geometric average option is cheaper than the arithmetic average option (AM-GM inequality).
- The MC geometric price matches the closed-form solution, validating our simulation.

> **Key Concept:** The price ordering European > Arithmetic Asian > Geometric Asian always holds. This ordering is a consequence of Jensen's inequality and the AM-GM inequality applied to option payoffs.

### The Price Hierarchy

$$\text{European} > \text{Arithmetic Asian} > \text{Geometric Asian}$$

This ordering always holds for options with the same strike and maturity. It is a consequence of Jensen's inequality applied through the convexity of the payoff function.

**Intuition:** Averaging "smooths out" the price path, making extreme outcomes less likely. Since options have asymmetric payoffs (you benefit from extreme highs but are protected from extreme lows), reducing extremes reduces the option value. The geometric mean smooths more aggressively than the arithmetic mean, so it produces even cheaper options.

> **Key Concept:** Asian options are cheaper than European options because averaging reduces the effective volatility of the payoff. Think of it this way: the average of 252 daily prices is much less volatile than a single end-of-year price, so the "option on the average" has less chance of a big payoff — and therefore costs less.

Notice also that the geometric average option is cheaper than the arithmetic average option. This is because the geometric mean is always less than or equal to the arithmetic mean (by the AM-GM inequality), so the geometric call payoff is systematically smaller.

## 7. Barrier Options: Knock-In and Knock-Out

### What Are Barrier Options?

A **barrier option** is like a regular option with an alarm system. If the stock price hits a predetermined level (the "barrier"), the option either:
- **Knocks out** (becomes worthless --- your option is "destroyed"), or
- **Knocks in** (becomes active --- it was dormant and is now "activated").

### The Four Types

| Type | Trigger | What Happens |
|------|---------|-------------|
| **Up-and-out** | Stock rises to hit barrier $B$ | Option dies (knocked out) |
| **Down-and-out** | Stock falls to hit barrier $B$ | Option dies |
| **Up-and-in** | Stock rises to hit barrier $B$ | Option activates (knocked in) |
| **Down-and-in** | Stock falls to hit barrier $B$ | Option activates |

### Real-World Analogy

Think of a "down-and-out" put as car insurance that cancels if you get into a minor fender-bender. You are still protected against a total loss, but if the car takes a small hit first (crossing the barrier), you lose your coverage. Because of this cancellation risk, the insurance is cheaper than a standard policy.

### The In-Out Parity

A beautiful result: for any barrier level $B$:

$$V_{\text{knock-in}} + V_{\text{knock-out}} = V_{\text{vanilla}}$$

Why? Every path either hits the barrier (activating the knock-in) or does not (keeping the knock-out alive). So one of the two barrier options is always "on." Together, they replicate the vanilla option exactly.

> **Key Concept:** In-out parity provides a powerful consistency check. If your knock-in + knock-out prices do not add up to the vanilla price, there is a bug in your code.

### Why Do Barrier Options Exist?

The primary reason is **cost reduction**. A knock-out option is always cheaper than the corresponding vanilla option because there are scenarios where the knock-out pays nothing but the vanilla would pay off. This makes barrier options popular with:

- **Corporate treasurers** who want cheap hedging and are willing to accept the knock-out risk
- **Structured product desks** who embed barriers to reduce the cost of embedded options
- **Retail investors** who buy "bonus certificates" and "turbo warrants" containing barrier features

### In-Out Parity: Worked Example

If a vanilla call is worth $\$10.45$ and the up-and-out call is worth $\$3.20$, then the up-and-in call must be worth $\$10.45 - \$3.20 = \$7.25$.

This works because every simulated path either touches the barrier (activating the knock-in and deactivating the knock-out) or does not touch the barrier (leaving the knock-out alive and the knock-in dormant). Exactly one of the pair is always active, so their payoffs sum to the vanilla payoff on every path.

> **CFA Exam Tip:** Barrier options are classified as exotic/path-dependent options. Know that they are cheaper than vanilla options, understand the four basic types (up/down + in/out), and know the in-out parity relationship.### The Four Types of Barrier Options

| Type | What happens | Common use |
|:-----|:------------|:-----------|
| **Up-and-out** | Dies if $S$ rises above $B$ | Cheaper call (give up upside beyond $B$) |
| **Up-and-in** | Activates if $S$ rises above $B$ | Bet on a breakout above $B$ |
| **Down-and-out** | Dies if $S$ falls below $B$ | Cheaper put (but disappears in a crash) |
| **Down-and-in** | Activates if $S$ falls below $B$ | Crash insurance that only kicks in when needed |

### The In-Out Parity

A beautiful no-arbitrage result:

$$\text{Knock-in} + \text{Knock-out} = \text{Vanilla}$$

This must hold for European options. If you hold both a knock-in and a knock-out with the same barrier, exactly one of them will be active at expiry — together they replicate the vanilla option exactly.

> **Common Mistake:** In-out parity only holds exactly for European-style exercise. For American barriers, the relationship becomes an inequality due to early exercise interactions.


Let's implement barrier option pricing via Monte Carlo. The key challenge is **path monitoring** — at each discrete time step, we check whether the stock price crossed the barrier. If it did, the option either activates (knock-in) or dies (knock-out):

> **Important:** Discrete monitoring means we might *miss* a barrier breach between time steps. The stock could cross the barrier intraday and come back, and our simulation wouldn't catch it. Finer time steps reduce this error but increase computation.

In [ ]:
def mc_barrier(S0, K, r, T, sigma, B, n_steps, n_paths, barrier_type='up-and-out',
               option_type='call', rng=rng):
    """Price a barrier option via Monte Carlo.
    
    barrier_type: 'up-and-out', 'down-and-out', 'up-and-in', 'down-and-in'
    """
    t, S = simulate_gbm_paths(S0, r, sigma, T, n_steps, n_paths, rng)
    
    S_max = S.max(axis=1)  # maximum price along each path
    S_min = S.min(axis=1)  # minimum price along each path
    ST = S[:, -1]          # terminal price
    
    # Determine which paths survive (are "alive" at expiry)
    if barrier_type == 'up-and-out':
        alive = S_max < B       # survived if never went above B
    elif barrier_type == 'down-and-out':
        alive = S_min > B       # survived if never went below B
    elif barrier_type == 'up-and-in':
        alive = S_max >= B      # activated if stock reached B
    elif barrier_type == 'down-and-in':
        alive = S_min <= B      # activated if stock fell to B
    else:
        raise ValueError(f"Unknown barrier type: {barrier_type}")
    
    # Payoffs (only for surviving/activated paths)
    if option_type == 'call':
        payoffs = np.maximum(ST - K, 0) * alive
    else:
        payoffs = np.maximum(K - ST, 0) * alive
    
    disc_payoffs = np.exp(-r * T) * payoffs
    return disc_payoffs.mean(), disc_payoffs.std() / np.sqrt(n_paths)

### Barrier Options: Common Pitfalls and Practical Notes

**Pitfall 1: The up-and-out call paradox.** An up-and-out call with a barrier just above the strike is almost worthless. To profit, the stock must rise above the strike (for the call payoff) but not above the barrier (to avoid knockout). If the barrier is close to the strike, this is nearly impossible. This is a classic "gotcha" for inexperienced traders.

**Pitfall 2: Hedging difficulty.** Near the barrier, the option's delta changes rapidly (the "barrier Greeks" are discontinuous). This makes hedging barrier options much harder than hedging vanilla options. A dealer who sells a barrier option must manage this hedging risk carefully.

**Pitfall 3: Gap risk.** If the stock jumps over the barrier (e.g., due to an overnight gap after earnings), the discrete-monitoring barrier may not trigger even though the stock effectively crossed the barrier level. This is a source of model risk.

> **Key Concept:** Barrier options are much more complex to trade and hedge than vanilla options, even though their payoff structure seems simple. The difficulty lies in the discontinuity at the barrier level, which creates extreme sensitivity in the option's Greeks.
> **CFA Exam Tip:** Barrier options are classified as "exotic" options. They are cheaper than vanilla options because the barrier condition removes some possible payoff scenarios. The more likely the barrier is to be hit, the cheaper the knock-out option (and the more expensive the knock-in).


Let us price barrier options and verify the in-out parity. We use both an up-barrier (above the current stock price) and a down-barrier (below the current stock price).

In [ ]:
# Price barrier options and verify in-out parity
n_paths = 200000
n_steps = 252
B_up = 120    # up barrier (above current price)
B_down = 80   # down barrier (below current price)

vanilla = bsm_call(S0, K, r, T, sigma)

uo_price, uo_se = mc_barrier(S0, K, r, T, sigma, B_up, n_steps, n_paths,
                              'up-and-out', 'call', rng=np.random.default_rng(42))
ui_price, ui_se = mc_barrier(S0, K, r, T, sigma, B_up, n_steps, n_paths,
                              'up-and-in', 'call', rng=np.random.default_rng(42))
do_price, do_se = mc_barrier(S0, K, r, T, sigma, B_down, n_steps, n_paths,
                              'down-and-out', 'call', rng=np.random.default_rng(42))
di_price, di_se = mc_barrier(S0, K, r, T, sigma, B_down, n_steps, n_paths,
                              'down-and-in', 'call', rng=np.random.default_rng(42))

print(f"Barrier Call Options (K={K}, T={T}, sigma={sigma})")
print(f"  Vanilla BSM:    {vanilla:.4f}")
print(f"\n  Up barrier B = {B_up}:")
print(f"    Up-and-out:   {uo_price:.4f} +/- {uo_se:.4f}")
print(f"    Up-and-in:    {ui_price:.4f} +/- {ui_se:.4f}")
print(f"    Sum (should = vanilla): {uo_price + ui_price:.4f}")
print(f"\n  Down barrier B = {B_down}:")
print(f"    Down-and-out: {do_price:.4f} +/- {do_se:.4f}")
print(f"    Down-and-in:  {di_price:.4f} +/- {di_se:.4f}")
print(f"    Sum (should = vanilla): {do_price + di_price:.4f}")

The in-out parity holds: knock-in + knock-out approximately equals the vanilla price (small differences are due to MC sampling error and discrete barrier monitoring).

Notice some interesting features:
- The **up-and-out call** with barrier 120 is quite cheap --- any path where the stock goes high enough to be profitable is also likely to hit the barrier and knock out. This creates a "catch-22" for the option holder.
- The **down-and-out call** with barrier 80 is close to the vanilla price --- the barrier is far from the money, so it rarely triggers.

> **Common Mistake:** Discrete monitoring (checking the barrier only at daily closes) versus continuous monitoring (checking at every instant) gives different prices. In practice, most barrier options specify discrete monitoring dates, but pricing them with continuous monitoring formulas can introduce significant bias.

### A Subtlety: Discrete vs Continuous Monitoring

In our simulation, we check the barrier only at discrete time steps (daily closes with 252 steps). In reality, the stock price is continuous, so it could hit the barrier between our monitoring dates. This introduces a **bias**: discrete monitoring always overprices knock-out options (and underprices knock-in options) because it misses some barrier breaches.

Broadie, Glasserman, and Kou (1997) developed a correction factor that adjusts for this discrete monitoring bias. For production-quality pricing, this correction is essential.

## 8. Convergence Analysis

The Monte Carlo estimator has well-understood statistical properties:

| Property | Value |
|----------|-------|
| **Bias** | Zero (it is an unbiased estimator) |
| **Standard error** | $\sigma_{\text{payoff}} / \sqrt{N}$ |
| **95% confidence interval** | $\hat{V} \pm 1.96 \times \text{SE}$ |
| **Convergence rate** | $O(1/\sqrt{N})$ regardless of dimension |

The last point is crucial: **MC convergence does not depend on the dimension of the problem**. For a 100-dimensional basket option (depending on 100 stocks), MC converges at the same $O(1/\sqrt{N})$ rate as for a single stock. This is why MC is the method of choice for high-dimensional derivatives.

Compare this with grid-based methods (finite differences, binomial trees), where the computational cost grows exponentially with dimension --- the "curse of dimensionality." Monte Carlo is immune to this curse.

> **Key Concept:** The dimension-independence of MC convergence is its superpower. For problems involving many underlying assets or many time steps, MC is often the only practical approach. This is why every major derivatives desk relies heavily on Monte Carlo simulation.### Confidence Intervals for MC Estimates

Because the MC estimator is an average of i.i.d. payoffs, the CLT guarantees it's approximately normal for large $N$. This gives us easy confidence intervals:

$$\hat{C} \pm z_{\alpha/2} \times SE$$

For a 95% CI: $\hat{C} \pm 1.96 \times SE$. For 99%: $\hat{C} \pm 2.576 \times SE$.

> **Key Concept:** The confidence interval is one of MC's biggest advantages over tree/PDE methods. With MC, you always know *how uncertain* your price estimate is. With 100 tree steps, you know the price but not how far it is from the true continuous-time value.


### Practical Tips for Production Monte Carlo

Based on industry practice, here are key guidelines for implementing MC pricing systems:

**1. Seed management.** Always use reproducible seeds for debugging, but use different seeds for production runs to avoid systematic biases.

**2. Parallelization.** MC is "embarrassingly parallel" --- each path is independent, so you can distribute paths across CPUs or GPUs trivially. Modern implementations use GPU computing (CUDA/OpenCL) for 100x speedups.

**3. Validation.** Always validate against known analytical solutions before pricing exotics. If your MC does not match BSM for vanilla Europeans, nothing else will be correct.

**4. Convergence diagnostics.** Plot running averages (as we did above) to visually check convergence. Also compare results from different random seeds to check stability.

**5. Bias vs. variance tradeoff.** Some payoffs introduce discretization bias (barrier options, early exercise). Increasing the number of time steps reduces bias but increases the cost per path. Find the right balance.

> **CFA Exam Tip:** While the CFA exam does not test implementation details, it does test the conceptual understanding that MC accuracy improves with more simulations, that path-dependent options require full path simulation, and that variance reduction techniques exist to improve efficiency.### Choosing the Right Number of Paths

A practical workflow:
1. **Start with 10,000 paths** to get a rough estimate and check your code
2. **Increase to 100,000** to get 2-decimal-place accuracy for most options
3. **Use 1,000,000+** for production pricing or when tight confidence intervals are needed
4. **Always report the standard error** — a price without an error bar is meaningless

> **Key Concept:** The standard error is not just a nice-to-have — it's integral to the MC estimate. A price of \$5.23 with SE of \$0.50 is very different from \$5.23 with SE of \$0.01. Always report both.


### Why Is MC Dimension-Independent?

The standard error depends only on $\sigma_{\text{payoff}}$ (the variability of the payoff) and $N$ (the number of samples). It does not depend on how many random numbers you need per sample. Whether each path requires 1 random number or 10,000, the convergence rate is the same --- only the cost per path increases.

This is why for a basket option on 50 stocks with daily path simulation over 1 year:
- Each path requires $50 \times 252 = 12{,}600$ random numbers.
- But the convergence rate is still $O(1/\sqrt{N})$ in the number of paths.
- A PDE or tree method would need a grid in 50 dimensions --- computationally impossible.

Compare this to a binomial tree, which in 1 dimension has $O(N)$ nodes for $N$ time steps but in $d$ dimensions has $O(N^d)$ nodes. For $d = 50$, even $N = 10$ gives $10^{50}$ nodes --- far more than the number of atoms in the universe.

Let's visualise the running convergence of a Monte Carlo estimate. This plot shows how the price estimate stabilises as we add more paths — early estimates are noisy, but they settle down:

In [ ]:
# Running convergence plot with confidence intervals
n_total = 100000
ST, _ = simulate_gbm_terminal(S0, r, sigma, T, n_total, rng=np.random.default_rng(42))
payoffs = np.exp(-r * T) * np.maximum(ST - K, 0)

# Compute running mean and CI as we add more paths one by one
cum_sum = np.cumsum(payoffs)
cum_sq_sum = np.cumsum(payoffs**2)
n_arr = np.arange(1, n_total + 1)

running_mean = cum_sum / n_arr
running_var = cum_sq_sum / n_arr - running_mean**2
running_var = np.maximum(running_var, 0)  # numerical safety
running_se = np.sqrt(running_var / n_arr)

fig, ax = plt.subplots(figsize=(12, 6))

# Subsample for plotting efficiency
idx = np.unique(np.logspace(0, np.log10(n_total), 2000).astype(int)) - 1
idx = idx[idx < n_total]

ax.plot(n_arr[idx], running_mean[idx], color=PRIMARY, linewidth=1, label='MC Estimate')
ax.fill_between(n_arr[idx],
                running_mean[idx] - 1.96 * running_se[idx],
                running_mean[idx] + 1.96 * running_se[idx],
                alpha=0.2, color=PRIMARY, label='95% CI')
ax.axhline(bsm_ref, color=SECONDARY, linestyle='--', linewidth=2, label=f'BSM = {bsm_ref:.4f}')
ax.set_xscale('log')
ax.set_xlabel('Number of Paths')
ax.set_ylabel('Price Estimate')
ax.set_title('Monte Carlo Running Convergence')
ax.legend()
plt.tight_layout()
plt.show()

The plot shows the MC estimate becoming increasingly stable as more paths are added. Early on (left side), the estimate jumps around wildly. By 100,000 paths, it has settled close to the BSM value with a narrow confidence band.

This running convergence plot is a standard diagnostic in practice. If the estimate is still moving significantly when you stop adding paths, you need more paths (or better variance reduction).

## 9. Summary: When to Use Monte Carlo

| Situation | Use MC? | Why |
|-----------|:---:|-----|
| Vanilla European option | No | BSM formula is exact and instant |
| Path-dependent exotic | **Yes** | MC handles any payoff naturally |
| High-dimensional (basket) | **Yes** | MC does not suffer from curse of dimensionality |
| American option | Maybe | Least-squares MC (Longstaff-Schwartz) works, but trees/PDE can be better |
| Need very high accuracy | Depends | MC is slow to converge; use variance reduction |
| Calibration (many re-pricings) | Depends | MC is slow per evaluation; consider analytical approximations |

> **Key Concept:** Monte Carlo is the Swiss Army knife of computational finance. It is rarely the most efficient method for any single problem, but it is almost always applicable. When all else fails, you can always simulate. The art is in making it fast enough through variance reduction, and knowing when an alternative method (analytical formula, PDE, tree) is better.

### Key Takeaways

1. **Risk-neutral pricing** is the foundation: simulate with drift $r$, not $\mu$, and discount at $r$.
2. **MC convergence** is $O(1/\sqrt{N})$ --- slow, but dimension-independent.
3. **Variance reduction** (antithetic variates, control variates) can dramatically improve efficiency without more paths.
4. **Asian options** are cheaper than vanilla because averaging reduces effective volatility. The arithmetic Asian has no closed form.
5. **Barrier options** have the elegant in-out parity: knock-in + knock-out = vanilla.
6. **Always report standard errors** --- an MC price without error bounds is meaningless.

> **CFA Exam Tip:** For the CFA exam, remember that Monte Carlo is the go-to method for path-dependent and multi-asset derivatives. Know its strengths (flexibility, dimension-independence), weaknesses (slow convergence, not directly applicable to American options without modification), and the basic variance reduction ideas.### Formula Reference Card

| Quantity | Formula |
|:---------|:--------|
| Risk-neutral terminal price | $S_T = S_0 \exp\left[(r - \sigma^2/2)T + \sigma\sqrt{T}\, Z\right]$ |
| MC price estimate | $\hat{C} = e^{-rT} \frac{1}{N}\sum_{i=1}^N \text{payoff}(S_T^{(i)})$ |
| Standard error | $SE = e^{-rT} \frac{\hat{\sigma}_{\text{payoff}}}{\sqrt{N}}$ |
| 95% confidence interval | $\hat{C} \pm 1.96 \times SE$ |
| Antithetic estimator | $\hat{C} = e^{-rT} \frac{1}{N}\sum_{i=1}^N \frac{f(Z_i) + f(-Z_i)}{2}$ |
| Asian payoff (arithmetic) | $\max\left(\frac{1}{M}\sum_{j=1}^M S_{t_j} - K,\; 0\right)$ |
| Barrier condition (up-out) | Option dies if $\max_j S_{t_j} \geq B$ |

> **Final Thought:** Monte Carlo is conceptually the simplest pricing method — "simulate and average." Its power comes from its generality: any payoff you can write down, you can price with MC. The art lies in doing it efficiently through variance reduction, and understanding the statistical properties of your estimate.


## 10. References

1. Glasserman, P. (2003). *Monte Carlo Methods in Financial Engineering*. Springer.
2. Boyle, P. P. (1977). *Options: A Monte Carlo approach*. Journal of Financial Economics, 4(3), 323-338.
3. Hull, J. C. (2018). *Options, Futures, and Other Derivatives* (10th ed.). Pearson.
4. Kemna, A. G. Z., & Vorst, A. C. F. (1990). *A pricing method for options based on average asset values*. Journal of Banking & Finance, 14(1), 113-129.
5. Broadie, M., Glasserman, P., & Kou, S. G. (1997). *A continuity correction for discrete barrier options*. Mathematical Finance, 7(4), 325-349.
6. Longstaff, F. A., & Schwartz, E. S. (2001). *Valuing American options by simulation: A simple least-squares approach*. Review of Financial Studies, 14(1), 113-147.

### Further Reading for Beginners

- Hull, J.C. *Options, Futures, and Other Derivatives*, Chapters 19-20 — accessible introduction to MC in finance.
- Wilmott, P. *Paul Wilmott Introduces Quantitative Finance*, Chapter 26 — intuitive presentation with worked examples.